# 1_Drop_Medallion

## Objetivo
Eliminar de forma controlada las tablas y sus archivos físicos de las capas Bronze, Silver y Golden, siguiendo la organización del notebook de limpieza desarrollado en clase.

**Importante:** este notebook es auxiliar de limpieza y reconstrucción. No forma parte de la ejecución normal de la ETL. La limpieza física es irreversible.

- Catálogo del proyecto: `dbw_proyecto_final_ehs`
- Contenedor de datos: `datalake`
- Capas: `bronze`, `silver` y `golden`
- Las tablas Bronze actuales son externas y sus archivos se encuentran en ADLS Gen2.


## 1. Parámetros de limpieza

Se utilizan widgets, como en clase, para parametrizar el Storage Account, el contenedor y el catálogo.

El parámetro `modoEjecucion` tiene dos valores:
- `SIMULAR`: muestra qué tablas y ubicaciones se limpiarían, sin borrar información.
- `EJECUTAR`: elimina las tablas registradas y sus archivos físicos cuando la ubicación pertenece a la carpeta de la capa correspondiente.

El valor predeterminado es `SIMULAR` para proteger los datos ante una ejecución accidental.

In [ ]:
# Reiniciar y definir los widgets del notebook.
dbutils.widgets.removeAll()

dbutils.widgets.text("nameContainer", "datalake", "Contenedor")
dbutils.widgets.text("nameStorage", "stproyectofinalehs", "Storage Account")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs", "Catálogo")
dbutils.widgets.dropdown("modoEjecucion", "SIMULAR", ["SIMULAR", "EJECUTAR"], "Modo de ejecución")

In [ ]:
# Obtener los parámetros y construir las rutas ADLS Gen2.
nameContainer = dbutils.widgets.get("nameContainer").strip()
nameStorage = dbutils.widgets.get("nameStorage").strip()
catalogo = dbutils.widgets.get("catalogo").strip()
modo_ejecucion = dbutils.widgets.get("modoEjecucion").strip().upper()

if not all([nameContainer, nameStorage, catalogo]):
    raise ValueError("Los parámetros de contenedor, Storage Account y catálogo son obligatorios.")

if modo_ejecucion not in ["SIMULAR", "EJECUTAR"]:
    raise ValueError("modoEjecucion debe ser SIMULAR o EJECUTAR.")

ruta_datalake = f"abfss://{nameContainer}@{nameStorage}.dfs.core.windows.net"

rutas_capas = {
    "bronze": f"{ruta_datalake}/bronze/",
    "silver": f"{ruta_datalake}/silver/",
    "golden": f"{ruta_datalake}/golden/"
}

print(f"Catálogo: {catalogo}")
print(f"Contenedor: {nameContainer}")
print(f"Storage Account: {nameStorage}")
print(f"Modo de ejecución: {modo_ejecucion}")
for capa, ruta in rutas_capas.items():
    print(f"{capa}: {ruta}")

## 2. Funciones de revisión y limpieza

Se consultan las tablas registradas en cada esquema mediante Unity Catalog. Para cada tabla se obtiene su ubicación física con `DESCRIBE DETAIL`.

La eliminación física solo se permitirá cuando la ubicación esté dentro de la carpeta ADLS de la capa que se está procesando. Así se evita borrar por error una ubicación administrada u otra ruta ajena al proyecto.

In [ ]:
# Funciones reutilizables para identificar tablas y validar sus ubicaciones.

def obtener_tablas_y_ubicaciones(esquema):
    tablas_df = spark.sql(f"SHOW TABLES IN `{catalogo}`.`{esquema}`")
    tablas = [fila["tableName"] for fila in tablas_df.collect()]

    resultado = []
    for nombre_tabla in tablas:
        nombre_completo = f"`{catalogo}`.`{esquema}`.`{nombre_tabla}`"
        detalle = spark.sql(f"DESCRIBE DETAIL {nombre_completo}").first()
        resultado.append({
            "tabla": nombre_tabla,
            "nombre_completo": nombre_completo,
            "ubicacion": detalle["location"]
        })

    return resultado


def validar_ubicacion_capa(ubicacion, ruta_capa):
    ubicacion_normalizada = ubicacion.rstrip("/") + "/"
    ruta_normalizada = ruta_capa.rstrip("/") + "/"
    return ubicacion_normalizada.startswith(ruta_normalizada)


def revisar_capa(esquema):
    elementos = obtener_tablas_y_ubicaciones(esquema)

    print(f"Esquema: {catalogo}.{esquema}")
    if not elementos:
        print("No existen tablas registradas.")
    else:
        for elemento in elementos:
            pertenece = validar_ubicacion_capa(
                elemento["ubicacion"],
                rutas_capas[esquema]
            )
            print(
                f"Tabla: {elemento['nombre_completo']} | "
                f"Ubicación: {elemento['ubicacion']} | "
                f"Ubicación de capa válida: {pertenece}"
            )
            if not pertenece:
                raise ValueError(
                    f"La limpieza se detiene antes de borrar objetos: "
                    f"{elemento['nombre_completo']} está fuera de la ruta de {esquema}."
                )

    return elementos

## 3. Eliminación de tablas Bronze

Se identifican dinámicamente las tablas registradas en `bronze`. Esto incluye las cinco tablas creadas por la ingesta actual: `almacenes`, `clientes`, `inventario`, `productos` y `ventas`.

Primero se elimina el registro de cada tabla en Unity Catalog mediante `DROP TABLE IF EXISTS`. La eliminación de archivos físicos se realiza en la siguiente celda, siguiendo el procedimiento visto en clase.

In [ ]:
# Revisar las tablas Bronze y sus ubicaciones antes de eliminarlas.
tablas_bronze = revisar_capa("bronze")

In [ ]:
# DROP TABLES (Bronze).
if modo_ejecucion == "EJECUTAR":
    for elemento in tablas_bronze:
        spark.sql(f"DROP TABLE IF EXISTS {elemento['nombre_completo']}")
        print(f"Tabla eliminada del catálogo: {elemento['nombre_completo']}")
else:
    print("SIMULACIÓN: no se eliminó ninguna tabla Bronze.")

In [ ]:
# REMOVE DATA (Bronze).
if modo_ejecucion == "EJECUTAR":
    for elemento in tablas_bronze:
        ubicacion = elemento["ubicacion"]

        if not validar_ubicacion_capa(ubicacion, rutas_capas["bronze"]):
            raise ValueError(
                f"Se canceló la limpieza: la ubicación de "
                f"{elemento['nombre_completo']} está fuera de Bronze: {ubicacion}"
            )

        dbutils.fs.rm(ubicacion, True)
        print(f"Archivos físicos eliminados: {ubicacion}")
else:
    print("SIMULACIÓN: no se eliminaron archivos físicos de Bronze.")

## 4. Eliminación de tablas Silver

Se aplica el mismo procedimiento de limpieza a las tablas que existan en el esquema `silver`. Actualmente este esquema no contiene tablas; cuando se desarrolle la transformación Silver, el notebook identificará sus tablas sin tener que modificar una lista manual.

In [ ]:
# Revisar las tablas Silver y sus ubicaciones antes de eliminarlas.
tablas_silver = revisar_capa("silver")

In [ ]:
# DROP TABLES (Silver).
if modo_ejecucion == "EJECUTAR":
    for elemento in tablas_silver:
        spark.sql(f"DROP TABLE IF EXISTS {elemento['nombre_completo']}")
        print(f"Tabla eliminada del catálogo: {elemento['nombre_completo']}")
else:
    print("SIMULACIÓN: no se eliminó ninguna tabla Silver.")

In [ ]:
# REMOVE DATA (Silver).
if modo_ejecucion == "EJECUTAR":
    for elemento in tablas_silver:
        ubicacion = elemento["ubicacion"]

        if not validar_ubicacion_capa(ubicacion, rutas_capas["silver"]):
            raise ValueError(
                f"Se canceló la limpieza: la ubicación de "
                f"{elemento['nombre_completo']} está fuera de Silver: {ubicacion}"
            )

        dbutils.fs.rm(ubicacion, True)
        print(f"Archivos físicos eliminados: {ubicacion}")
else:
    print("SIMULACIÓN: no se eliminaron archivos físicos de Silver.")

## 5. Eliminación de tablas Golden

Se identifican y limpian las tablas registradas en el esquema `golden`. Actualmente el esquema está vacío; las futuras tablas Gold se incorporarán automáticamente a este proceso de limpieza.

In [ ]:
# Revisar las tablas Golden y sus ubicaciones antes de eliminarlas.
tablas_golden = revisar_capa("golden")

In [ ]:
# DROP TABLES (Golden).
if modo_ejecucion == "EJECUTAR":
    for elemento in tablas_golden:
        spark.sql(f"DROP TABLE IF EXISTS {elemento['nombre_completo']}")
        print(f"Tabla eliminada del catálogo: {elemento['nombre_completo']}")
else:
    print("SIMULACIÓN: no se eliminó ninguna tabla Golden.")

In [ ]:
# REMOVE DATA (Golden).
if modo_ejecucion == "EJECUTAR":
    for elemento in tablas_golden:
        ubicacion = elemento["ubicacion"]

        if not validar_ubicacion_capa(ubicacion, rutas_capas["golden"]):
            raise ValueError(
                f"Se canceló la limpieza: la ubicación de "
                f"{elemento['nombre_completo']} está fuera de Golden: {ubicacion}"
            )

        dbutils.fs.rm(ubicacion, True)
        print(f"Archivos físicos eliminados: {ubicacion}")
else:
    print("SIMULACIÓN: no se eliminaron archivos físicos de Golden.")

## 6. Validación posterior a la limpieza

Se vuelve a consultar Unity Catalog y las ubicaciones físicas de las tres capas para comprobar el resultado.

En modo `SIMULAR`, las tablas deben continuar registradas. En modo `EJECUTAR`, los esquemas deben quedar sin tablas y las carpetas de las tablas eliminadas ya no deben contener sus datos.

In [ ]:
# Validar que no queden tablas registradas en los esquemas Medallion.
for esquema in ["bronze", "silver", "golden"]:
    print(f"\nTablas restantes en {catalogo}.{esquema}:")
    display(spark.sql(f"SHOW TABLES IN `{catalogo}`.`{esquema}`"))

In [ ]:
# Validar el contenido físico de las carpetas Medallion.
for capa, ruta in rutas_capas.items():
    print(f"\nContenido físico de {capa}: {ruta}")

    try:
        display(dbutils.fs.ls(ruta))
    except Exception as e:
        mensaje = str(e)
        if "PathNotFound" in mensaje or "The specified path does not exist" in mensaje:
            print(f"La carpeta {capa} no existe o quedó vacía tras la limpieza.")
        else:
            raise

## 7. Resumen de ejecución

La limpieza de Bronze, Silver y Golden ha finalizado según el modo seleccionado.

- `SIMULAR`: no modifica tablas ni archivos; permite revisar qué se limpiaría.
- `EJECUTAR`: elimina las tablas registradas y sus archivos físicos dentro de las rutas de cada capa.

**Advertencia:** antes de seleccionar `EJECUTAR`, verifica que estás trabajando en el catálogo y Storage Account del proyecto. No ejecutes este notebook como parte del flujo ETL habitual.

In [ ]:
print("RESUMEN DE DROP MEDALLION")
print("-------------------------")
print(f"Catálogo: {catalogo}")
print(f"Modo de ejecución: {modo_ejecucion}")
print(f"Tablas identificadas en Bronze: {len(tablas_bronze)}")
print(f"Tablas identificadas en Silver: {len(tablas_silver)}")
print(f"Tablas identificadas en Golden: {len(tablas_golden)}")

if modo_ejecucion == "SIMULAR":
    print("Simulación completada. No se eliminaron datos.")
else:
    print("Proceso de limpieza ejecutado. Revisar las validaciones anteriores.")